In [30]:
import os
import google.generativeai as genai
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import json, time, os
from pathlib import Path
from google.api_core.exceptions import ResourceExhausted
import re
import math
import ast
from collections import Counter
import urllib3
import pypdf

In [10]:
from google.colab import userdata
import google.generativeai as genai

genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

model = genai.GenerativeModel('gemini-2.5-flash')

In [11]:


def extract_eligibility_from_html(html):
    """
    Try to extract 'eligibility-like' text from a grant page.
    If we can't find specific eligibility sections, fall back to the first paragraphs,
    and finally to whole-page text (trimmed).
    """
    soup = BeautifulSoup(html, "html.parser")

    text_blocks = []

    # 1) Look for headings that mention eligibility / requirements
    headers = soup.find_all(["h1", "h2", "h3", "h4"])
    keywords = ["eligib", "eligibility", "who can apply", "applicant", "requirements", "who is eligible"]

    for h in headers:
        htxt = h.get_text(separator=" ").lower()
        if any(k in htxt for k in keywords):
            block = h.get_text(separator=" ").strip()

            # Collect a few following siblings (paragraphs/lists) after the heading
            sib = h.find_next_sibling()
            collected = []
            steps = 0
            while sib is not None and steps < 5:
                # Only collect "content-ish" elements
                if sib.name in ["p", "div", "ul", "ol", "li"]:
                    collected.append(sib.get_text(separator=" ").strip())
                sib = sib.find_next_sibling()
                steps += 1

            if collected:
                block = block + "\n" + "\n".join(collected)
            text_blocks.append(block)

    # 2) Fallback: if we didn't find any useful headings, use the first few paragraphs
    if not text_blocks:
        paras = soup.find_all("p")
        if paras:
            first_paras = [p.get_text(separator=" ").strip() for p in paras[:6]]
            text_blocks.append("\n".join(first_paras))

    # 3) Final fallback: use trimmed whole-page text
    if not text_blocks:
        whole = soup.get_text(separator=" ", strip=True)
        text_blocks.append(whole[:2000])

    # Join all blocks, filter out empties
    result = "\n\n".join(block for block in text_blocks if block).strip()
    return result


In [12]:
def google_cse_search(query,num=5):
    url = "https://customsearch.googleapis.com/customsearch/v1"
    params = {
        "key": os.environ["G_API_KEY"],
        "cx": "d6f74faa9b39f4448",
        "q": query,
        "num": num
    }
    r = requests.get(url, params=params, timeout=10)
    data = r.json()

    urls = []
    return [i.get("link") for i in data.get("items", []) if i.get("link")]

In [13]:
def looks_like_travel_grant(url, text):
    u = url.lower()
    t = text.lower()

    # 1. Block obvious non-grant domains
    BAD_DOMAINS = [
        "weforum.org", "linkedin.com", "facebook.com", "twitter.com",
        "youtube.com", "instagram.com", "prnewswire.com",
        "news", "/news/", "blog", "/blog/"
    ]
    if any(b in u for b in BAD_DOMAINS):
        return False

    # 2. Hard ACCEPT if URL explicitly mentions a grant
    grant_terms_url = [
        "grant", "funding", "fellowship", "scholarship",
        "travel-award", "travelgrant", "travel-grants",
        "mobility-fund"
    ]
    if any(term in u for term in grant_terms_url):
        return True

    # 3. Look for positive grant indicators in text
    POSITIVE = [
        "eligibility", "who can apply", "funding amount",
        "travel support", "reimbursement", "application deadline",
        "award", "grant", "fund", "fellowship", "scholarship",
        "travel grant", "travel funding", "mobility fund",
        "supports travel", "airfare", "conference travel"
    ]
    if sum(t.count(k) for k in POSITIVE) >= 2:
        return True

    # 4. Reject if page is purely informational with zero grant markers
    return False

In [14]:
def detect_country_from_text(text):
    countries = [
        "India", "UAE", "United States", "USA", "UK", "United Kingdom",
        "Canada", "Germany", "France", "Japan", "China", "Singapore",
        "Australia", "Netherlands", "Switzerland", "Sweden", "Italy",
        "Spain", "Brazil", "South Korea", "Israel"
    ]

    for c in countries:
        if c.lower() in text.lower():
            return c
    return None

In [15]:
def detect_profile_mode(profile, pdf_text, url_text):
    """
    VERY IMPORTANT:
    Determines what type of grant search we must perform.
    """

    combined = (pdf_text + " " + url_text).lower()

    academic_terms = ["conference", "symposium", "abstract", "presentation", "poster", "session"]
    funding_terms = ["research", "study", "project", "fellowship"]

    if any(t in combined for t in academic_terms):
        return "ACADEMIC"

    if any(t in combined for t in funding_terms):
        return "RESEARCH"

    return "GENERIC"

In [16]:
def compute_match_score(profile, summary_text):
    """
    Scores how likely this URL represents a REAL academic travel grant
    for presenting research at an international conference.
    """

    if not summary_text:
        return 0.0

    text = summary_text.lower()
    score = 0

    # 1. Check for key travel-grant signals (VERY IMPORTANT)
    travel_keywords = [
        "travel grant", "international travel", "travel support",
        "present paper", "present research", "conference travel",
        "airfare", "visa support", "registration fee",
        "reimbursement", "its", "serb", "anrf", "csir",
        "icmr", "iccr", "ugc", "mobility grant", "fellowship",
    ]

    score += sum(1 for w in travel_keywords if w in text) * 10

    # 2. Must be about FUNDING
    funding_terms = ["funding", "grant", "support", "financial", "fellowship"]
    score += sum(1 for w in funding_terms if w in text) * 5

    # 3. Must apply to researchers / students
    researcher_terms = ["researcher", "student", "faculty", "scholar", "phd", "postdoc"]
    score += sum(1 for w in researcher_terms if w in text) * 3

    # 4. Must allow India (origin location)
    locs = profile.get("locations", [])
    for loc in locs:
        if loc.lower() in text:
            score += 8

    # 5. Mission match (very light weight)
    mission = profile.get("mission", "").lower()
    for word in mission.split():
        if len(word) > 5 and word in text:
            score += 0.8

    # Cap score at 100 max
    return min(score, 100)

In [17]:

def bing_search(query, max_results=5):
    try:
        url = "https://www.bing.com/search"
        params = {"q": query, "form": "QBLH"}

        headers = {
            "User-Agent":
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0 Safari/537.36"
        }

        r = requests.get(url, params=params, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        results = []

        # Bing result selectors
        for item in soup.select("li.b_algo h2 a")[:max_results]:
            href = item.get("href")
            if href and href.startswith("http"):
                results.append(href)

        # In case Bing changes layout
        if not results:
            for item in soup.select("a")[:max_results * 2]:
                href = item.get("href")
                if href and href.startswith("http"):
                    results.append(href)

        return list(dict.fromkeys(results))  # unique results

    except Exception as e:
        print(f"⚠️ Bing fallback error: {e}")
        return []

In [18]:
STOPWORDS = set("""
a an the and or in on for of to is are was were be been being this that these those with from by at as into about after before under over again further once
""".split())

In [19]:


def extract_keywords(text, top_n=10):
    """
    Extracts domain keywords from any text:
    - Removes stopwords
    - Keeps meaningful technical terms
    - TF-IDF style weighting (frequency + length)
    """
    if not text:
        return []

    text = re.sub(r"[^a-zA-Z0-9 ]", " ", text)
    words = text.lower().split()

    words = [w for w in words if w not in STOPWORDS and len(w) > 4]

    if not words:
        return []

    freq = Counter(words)

    # weight = frequency * log(word_length)
    weighted = {
        word: freq[word] * math.log(len(word) + 1)
        for word in freq
    }

    sorted_words = sorted(weighted, key=weighted.get, reverse=True)
    return sorted_words[:top_n]


In [20]:
def make_queries_from_profile(profile):
    """
    Fully upgraded dynamic query generator:
    - Uses PDF + URL content
    - Extracts real keywords automatically
    - Detects mode (ACADEMIC / RESEARCH / GENERIC)
    - Creates international queries (not India-only)
    """

    # 1. Retrieve texts
    pdf_text = profile.get("pdf_text", "")
    url_text = profile.get("url_text", "")
    mission = profile.get("mission", "")
    user_keywords = profile.get("keywords", [])
    locations = profile.get("locations", [])

    # 2. Keyword extraction from source documents
    pdf_keywords = extract_keywords(pdf_text, top_n=8)
    url_keywords = extract_keywords(url_text, top_n=8)

    domain_keywords = list(set(pdf_keywords + url_keywords + [w.lower() for w in user_keywords]))

    # 3. Detect mode
    mode = detect_profile_mode(profile, pdf_text, url_text)
    print(f"🌎 Detected mode: {mode}")

    # 4. Core base queries
    BASE_ACADEMIC = [
        "academic travel grant",
        "international conference travel funding",
        "researcher travel support",
        "travel grant for presenting research",
        "global academic mobility grant",
    ]

    BASE_RESEARCH = [
        "international research grant",
        "global R&D funding",
        "researcher mobility scholarship",
        "international innovation grant",
    ]

    BASE_GENERIC = [
        "international grant program",
        "global funding opportunities",
        "scholarship and grant directory",
        "nonprofit funding opportunities",
    ]

    if mode == "ACADEMIC":
        base_queries = BASE_ACADEMIC
    elif mode == "RESEARCH":
        base_queries = BASE_RESEARCH + BASE_ACADEMIC
    else:
        base_queries = BASE_GENERIC + BASE_RESEARCH

    # 5. Domain-specific dynamic queries
    domain_queries = []
    for kw in domain_keywords:
        domain_queries.extend([
            f"international {kw} travel grant",
            f"{kw} research funding worldwide",
            f"global scholarship for {kw} research",
            f"grant to present {kw} research",
        ])

    # 6. Mission-based queries
    mission_queries = [
        f"funding to support {mission}",
        f"travel grant for {mission}",
        f"research funding related to {mission}",
    ]

    # 7. Location-specific (global, not India-only)
    location_queries = []
    for loc in locations:
        location_queries.append(f"{loc} researcher international travel funding")
        location_queries.append(f"travel scholarship for researchers from {loc}")

    # 8. Combine everything and dedupe
    all_queries = base_queries + domain_queries + mission_queries + location_queries

    final_queries = list({q.strip() for q in all_queries if len(q.strip()) > 10})

    print(f"🔍 Total queries generated: {len(final_queries)}")

    return final_queries

In [21]:



class SearchAgent:
    """
    FIXED VERSION — ultra-clean grant URL collector.
    - Uses Google CSE (your version)
    - Bing fallback
    - Filters by domain category + URL keywords
    """

    GOOD_KEYWORDS = [
        "grant", "fund", "funding", "travel", "conference",
        "fellowship", "scholarship", "mobility", "support"
    ]

    BAD_DOMAINS = [
        "facebook.com", "instagram.com", "twitter.com",
        "reddit.com", "youtube.com", "kraken.com",
        "weforum.org", "swift.com", "mckinsey.com",
        "imf.org", "projects.worldbank.org",
        "pmc.ncbi.nlm.nih.gov", "pubmed.ncbi.nlm.nih.gov",
        "sciencedirect.com", "nature.com",
        "meetings", "agenda", "abstract", "program"
    ]

    GOOD_DOMAINS = [
        "gov", "gov.in", "edu", "ac.in",
        "nsf.gov", "nih.gov", "who.int",
        "europa.eu", "ukri.org", "marie-sklodowska-curie",
        "daad.de", "royalsociety", "wellcome",
        "serb", "dst", "csir", "icmr", "anrf", "dbt",
        "foundation", "fellowship", "travel", "fund"
    ]

    def url_is_valid(self, url):
        u = url.lower()

        # Block obviously irrelevant or toxic
        if any(b in u for b in self.BAD_DOMAINS):
            return False

        # MUST contain at least one funding keyword
        if not any(k in u for k in self.GOOD_KEYWORDS):
            return False

        # Must come from at least tolerable domain
        if not any(d in u for d in self.GOOD_DOMAINS):
            return False

        return True

    def run(self, profile):
        queries = make_queries_from_profile(profile)
        collected = []

        for q in queries:
            print(f"\n🔍 Query: {q}")

            # Google CSE
            try:
                google_urls = google_cse_search(q)
                for u in google_urls:
                    if self.url_is_valid(u):
                        collected.append(u)
            except:
                pass

            # Bing fallback
            try:
                bing_urls = bing_search(q, max_results=5)
                for u in bing_urls:
                    if self.url_is_valid(u):
                        collected.append(u)
            except:
                pass

        # De-duplicate
        final = list(dict.fromkeys(collected))
        print(f"\n🔗 Valid grant URLs found: {len(final)}")

        return final



# Disable SSL warnings (for sites with broken HTTPS)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
SKIP_DOMAINS  = [
                            "linkedin.com",
                            "instagram.com",
                            "facebook.com",
                            "twitter.com",
                            "youtube.com",
                            "kraken.com",
                            "fatf-gafi.org",
                            "mckinsey.com",
                            "weforum.org",
                            "swift.com",
                            "imf.org",
                            "projects.worldbank.org",
                            "pmc.ncbi.nlm.nih.gov",
                            "pubmed.ncbi.nlm.nih.gov",
                            "science.org",
                            "sciencedirect.com",
                        ]
def _extract_eligibility_snippet(soup_text):
    """
    Try to find an 'eligibility' or related section in the plain text.
    Heuristics: look for headings/phrases and return a nearby slice.
    """
    low = soup_text.lower()
    # candidate anchors
    anchors = [
        "eligib", "who can apply", "who may apply", "eligibility criteria",
        "selection criteria", "who is eligible", "requirements", "applicant",
        "how to apply", "application requirements", "funding provided",
        "support provided", "award amount", "what we fund"
    ]
    for a in anchors:
        idx = low.find(a)
        if idx != -1:
            # return ~800 chars around the anchor (try to be helpful)
            start = max(0, idx - 200)
            end = start + 1200
            return soup_text[start:end].strip()
    # fallback: return first 1200 chars
    return soup_text[:1200].strip()


class ExtractorAgent:
    def __init__(self, skip_domains=None):
        self.skip_domains = skip_domains or SKIP_DOMAINS

    def fetch(self, url):
        # If URL is from forbidden domain → skip early
        if any(dom in url for dom in self.skip_domains):
            print(f"⏭️ Skipping blocked domain: {url}")
            return None, None  # (html_text, content_type)

        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/123.0 Safari/537.36"
            ),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
            "Accept-Encoding": "gzip, deflate, br",
            "Referer": "https://www.google.com/",
            "Connection": "keep-alive"
        }

        try:
            session = requests.Session()
            adapter = requests.adapters.HTTPAdapter(max_retries=2)
            session.mount("https://", adapter)
            session.mount("http://", adapter)

            r = session.get(url, headers=headers, timeout=30, verify=False)

            if r.status_code != 200:
                print(f"⚠️ Failed {url} (status {r.status_code})")
                return None, None

            content_type = r.headers.get("Content-Type", "").lower()
            # PDF handling
            if "pdf" in content_type or url.lower().endswith(".pdf"):
                try:
                    # read from bytes
                    pdf_bytes = io.BytesIO(r.content)
                    reader = pypdf.PdfReader(pdf_bytes)
                    text_pages = []
                    for p in reader.pages:
                        try:
                            page_text = p.extract_text() or ""
                            text_pages.append(page_text)
                        except Exception:
                            continue
                    full_text = "\n".join(text_pages).strip()
                    if not full_text:
                        print(f"⏭️ PDF had no extractable text: {url}")
                        return None, None
                    return full_text, "application/pdf"
                except Exception as e:
                    print(f"⚠️ PDF parse error for {url}: {e}")
                    return None, None

            # Non-HTML or non-text fallback
            if "text/html" not in content_type and "text" not in content_type:
                # some sites return empty or odd content types; skip
                print(f"⏭️ Non-HTML content skipped: {url} ({content_type})")
                return None, None

            return r.text, "text/html"

        except Exception as e:
            print(f"⚠️ Error fetching/parsing {url}: {e}")
            return None, None


    def run(self, urls):
        """
        Returns list of dicts with keys: { "url": ..., "text": ... }
        Ensures SummarizerAgent receives 'text'.
        """
        results = []

        for url in urls:
            html_or_text, ctype = self.fetch(url)
            if not html_or_text:
                continue

            if ctype == "application/pdf":
                # html_or_text is full text already from PDF extraction
                page_text = html_or_text
            else:
                # parse HTML to plain text and attempt to extract eligibility snippet
                soup = BeautifulSoup(html_or_text, "html.parser")

                # remove scripts, styles to reduce noise
                for s in soup(["script", "style", "noscript", "iframe"]):
                    s.decompose()

                plain = soup.get_text(" ", strip=True)
                if not plain or len(plain) < 200:
                    # too small, skip
                    print(f"⏭️ Extracted very little text, skipping: {url}")
                    continue

                # Try to pick eligibility-like snippet; fallback to full text
                snippet = _extract_eligibility_snippet(plain)
                page_text = snippet if snippet else plain[:2000]

            # Grant classifier filter uses the full page/plain text (not just snippet)
            # pass in the full plain text for classifier decision
            full_plain = page_text if ctype == "application/pdf" else plain

            if not looks_like_travel_grant(url, full_plain):
                print(f"🚫 Rejected non-grant page: {url}")
                continue

            print(f"✅ Accepted grant page: {url}")
            # Ensure key is "text" for SummarizerAgent
            results.append({
                "url": url,
                "text": page_text
            })

        return results




class SummarizerAgent:
    """Summarizes eligibility using Gemini, with quota-safe fallbacks.
    Defensive: ensures every summary has a numeric 'score' so sorting never fails.
    """

    def run(self, items, profile, max_llm_items=3):
        summaries = []
        if not items:
            return summaries

        # Use Gemini only on the first few items to stay within quota
        llm_items = items[:max_llm_items]
        fallback_items = items[max_llm_items:]

        for idx, it in enumerate(llm_items, start=1):
            raw_text = (it.get("text") or it.get("eligibility") or "").strip()
            if not raw_text:
                continue

            # safety: trim very long pages
            raw_text = raw_text[:4000]

            prompt = f"""
You are a grant-analysis assistant.

Summarize the following grant eligibility section in 5–7 bullet points.
Focus on:
- Who can apply
- Geographic focus
- Thematic focus
- Important restrictions

Text:
\"\"\"{raw_text}\"\"\"
"""

            summary = None
            try:
                print(f"🧠 LLM summarizing item {idx}/{len(llm_items)}")
                if "model" in globals() and hasattr(model, "generate_content"):
                    resp = model.generate_content(prompt)
                    summary = (getattr(resp, "text", None) or str(resp)).strip()
                else:
                    # No model available in this environment — force fallback
                    summary = None
            except ResourceExhausted:
                # Quota exhausted → stop calling Gemini and fall back
                print("⚠️ Gemini quota reached in SummarizerAgent. Using local fallback summaries for remaining items.")
                summary = None
                # Treat all remaining (including rest of llm_items) as fallback_items
                fallback_items = items[idx:]
            except Exception as e:
                # Any other error -> fallback for this item and continue
                print(f"⚠️ Error summarizing one item, using fallback. Details: {e}")
                summary = None

            # If LLM didn't produce a usable summary, use a safe fallback
            if not summary:
                summary = raw_text[:800]

            # Compute a numeric score defensively
            try:
                score_val = compute_match_score(profile, summary)
                # ensure float and clamp
                score = float(score_val) if score_val is not None else 0.0
            except Exception:
                score = 0.0

            # append a well-formed dict
            summaries.append({
                "url": it.get("url"),
                "summary": summary,
                "score": score
            })

        # Cheap local “summary” for remaining items (no LLM calls)
        for it in fallback_items:
            raw_text = (it.get("text") or "").strip()
            if not raw_text:
                continue
            summary = raw_text[:800]  # crude truncation
            try:
                score_val = compute_match_score(profile, summary)
                score = float(score_val) if score_val is not None else 0.0
            except Exception:
                score = 0.0

            summaries.append({
                "url": it.get("url"),
                "summary": summary,
                "score": score
            })

        # Rank by numeric score (defensive get with default 0.0)
        summaries.sort(key=lambda x: x.get("score", 0.0), reverse=True)
        print(f"✅ SummarizerAgent produced {len(summaries)} summaries.")
        return summaries



class DraftAgent:
    """Produces the grant proposal draft using Gemini, with clean, non-chatty output."""

    def run(self, top_grant, profile):
        # Handle missing / empty summary safely
        grant_summary = (top_grant.get("summary") or "").strip()
        if not grant_summary:
            grant_summary = (
                "No detailed grant description is available. "
                "Write a realistic, generic grant proposal aligned with the organization's mission and funding needs."
            )

        prompt = f"""
You are a professional NGO grant writer.

TASK:
Write a complete, well-structured grant proposal for the following organization.
Use the mission, locations, and grant summary to tailor the proposal.

IMPORTANT OUTPUT RULES:
- Output ONLY the proposal text.
- Do NOT include any explanations, comments, or meta-text.
- Do NOT say things like "Okay", "This is an excellent challenge", or mention that a summary was missing.
- Start directly with the proposal title or first heading.

Organization name: {profile['name']}
Mission: {profile['mission']}
Funding need (USD): {profile['budget_need_usd']}
Locations: {", ".join(profile.get("locations", []))}

Grant summary:
\"\"\"{grant_summary[:1500]}\"\"\"
"""

        response = model.generate_content(prompt)
        draft = (response.text or "").strip()

        # Post-process: strip any chatty meta-intro line if the model still misbehaves
        lines = draft.splitlines()
        while lines and any(
            bad in lines[0].lower()
            for bad in [
                "okay", "challenge", "since no", "i will", "as an ai",
                "this is an excellent", "i need to", "i will create"
            ]
        ):
            lines = lines[1:]
        clean_draft = "\n".join(lines).strip()

        return clean_draft


In [22]:

def clean_and_parse_llm_dict(raw):
    """
    Cleans LLM output so it becomes valid Python dict syntax.
    Handles:
    - `org_profile = {...}`
    - comments (# ...)
    - ellipses (...)
    - backticks and fences
    - trailing commas
    """

    # Remove code fences
    raw = raw.replace("```python", "").replace("```", "").strip()

    # Remove "org_profile ="
    raw = re.sub(r"^org_profile\s*=\s*", "", raw)

    # Remove inline comments starting with #
    raw = re.sub(r"#.*", "", raw)

    # Remove ellipses
    raw = raw.replace("...", "")

    # Remove trailing commas before closing braces or brackets
    raw = re.sub(r",\s*([}\]])", r"\1", raw)

    # Strip leading/trailing whitespace
    raw = raw.strip()

    # TRY PARSING SAFELY
    try:
        return ast.literal_eval(raw)
    except Exception as e:
        print("❌ Parsing still failed:", e)
        print("Cleaned text was:\n", raw)
        return {}

In [23]:



def generate_org_profile_from_sources(pdf_path, url, additional_needs=""):
    """
    Full pipeline:
    - Extracts text from PDF
    - Extracts text from URL
    - Includes user-specified additional needs (optional)
    - Calls LLM to create org_profile dict (ready for Search/Summarizer/Draft)
    """

    # ---------------------------
    # 1. PDF Extraction
    # ---------------------------
    pdf_text = ""
    try:
        reader = pypdf.PdfReader(pdf_path)
        for page in reader.pages:
            try:
                page_text = page.extract_text()
                if page_text:
                    pdf_text += page_text + "\n"
            except Exception:
                continue
        pdf_text = pdf_text[:7000]   # safety limit
    except Exception as e:
        print(f"⚠️ PDF extraction failed for '{pdf_path}': {e}")
        pdf_text = ""

    # ---------------------------
    # 2. URL Extraction
    # ---------------------------
    url_text = ""
    try:
        r = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=15
        )
        soup = BeautifulSoup(r.text, "html.parser")
        url_text = soup.get_text(" ", strip=True)[:5000]
    except Exception as e:
        print(f"⚠️ URL extraction failed for '{url}': {e}")
        url_text = ""

    # ---------------------------
    # 3. Build LLM Prompt
    # ---------------------------
    prompt = f"""
You are an academic funding assistant.

TASK:
Generate a clean Python dictionary called `org_profile` representing
a researcher seeking travel funding to attend an academic conference.

All details must be extracted from the REAL content below.

-----------------------------------------
PDF CONTENT (abstract / proposal / acceptance letter)
-----------------------------------------
\"\"\"{pdf_text}\"\"\"

-----------------------------------------
WEBSITE CONTENT (conference homepage or CFP)
-----------------------------------------
\"\"\"{url_text}\"\"\"

-----------------------------------------
ADDITIONAL USER-SPECIFIED NEEDS
-----------------------------------------
\"\"\"{additional_needs}\"\"\"

-----------------------------------------
INSTRUCTION: EXTRACT EVERYTHING FROM SOURCE MATERIAL
-----------------------------------------
Using ONLY the information from the PDF and URL:

1. Identify:
   - Researcher's name (if found)
   - Researcher's academic role
   - Conference name + theme
   - Conference location
   - Research field
   - Purpose of travel
   - Any explicit funding statements

2. If something is missing:
   - Infer from context
   - Estimate missing values responsibly

-----------------------------------------
MONEY ESTIMATION RULE
-----------------------------------------
Estimate `budget_need_usd` based on:
- airfare (origin → conference location)
- stay
- registration fees
- local travel
- food
- visa

locations:
    MUST be a Python list of strings only.
    Format:
        ["Origin Country", "Origin City", "Conference City", "Conference Country"]
    NEVER use a dictionary for locations.
-----------------------------------------
OUTPUT REQUIREMENTS
-----------------------------------------
- Output ONLY valid Python dict syntax.
- Include EXACTLY these fields:
    - name
    - type: "individual_researcher"
    - role
    - mission
    - objective
    - budget_need_usd
    - locations
    - keywords

IMPORTANT:
- DO NOT hardcode anything.
- Everything must come from extracted text or reasonable inference.
"""

    # ---------------------------
    # 4. Call LLM
    # ---------------------------
    resp = model.generate_content(prompt)
    raw = resp.text.strip()

    # Remove ```python or ``` blocks if model outputs markdown
    if raw.startswith("```"):
        raw = raw.strip("`")
        raw = raw.replace("python", "")
        raw = raw.strip()

    # ---------------------------
    # 5. Parse safely using ast.literal_eval
    # ---------------------------
    try:
        profile = clean_and_parse_llm_dict(raw)
        return profile
    except Exception as e:
        print("⚠️ Could not parse LLM output as dict:", e)
        print("Raw output was:")
        print(raw)
        return {}

In [24]:
org_profile = generate_org_profile_from_sources(
    pdf_path=r"/content/AIMS-CCU.pdf",
    url="https://www.aims2025.com/",
    additional_needs="this is purely an Abstract submission. The submitting person is Saksham Walia."
)


In [25]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
print(org_profile)

{'name': 'Saksham Walia', 'type': 'individual_researcher', 'role': 'Graduate Student Researcher', 'mission': 'To present research findings at an international academic conference.', 'objective': "To secure travel funding to attend and present the abstract 'AI-Accelerated Discovery of Materials for Carbon Capture and Utilization (CCU)' at the International Conference on AI and Materials for Sustainability.", 'budget_need_usd': 2600, 'locations': ['India', 'New Delhi', 'Abu Dhabi', 'United Arab Emirates'], 'keywords': ['AI', 'Materials Discovery', 'Carbon Capture and Utilization', 'CCU', 'Sustainability', 'Decarbonization', 'Machine Learning', 'Quantum Simulation', 'Nanoporous Frameworks', 'MOFs', 'COFs', 'Catalysis', 'Energy', 'Climate Action', 'Sustainable Development Goals']}


In [ ]:
# org_profile = {
#     "name": "Sunrise Community Health",
#     "mission": "Improve rural maternal health and community healthcare.",
#     "budget_need_usd": 25000,
#     "locations": ["Rajasthan", "India"],
#     "keywords": ["maternal health", "rural", "community"]
# }


In [27]:
search_agent = SearchAgent()
extractor_agent = ExtractorAgent()
summarizer_agent = SummarizerAgent()
draft_agent = DraftAgent()


print("🔍 Searching for grants...")
urls = search_agent.run(org_profile)

print("\n🔗 URLs found by SearchAgent:")
for i, u in enumerate(urls, start=1):
    print(f"{i}. {u}")



print("\n📄 Extracting eligibility...")
extracted = extractor_agent.run(urls)

# Optional: limit how many pages to process deeply
extracted = extracted[:10]

print("\n🧠 Summarizing and Scoring...")
summaries = summarizer_agent.run(extracted, org_profile, max_llm_items=3)

top = summaries[0] if summaries else {"url": None, "summary": "", "score": 0}

print("\n✍ Generating Proposal Draft...")
draft = draft_agent.run(top, org_profile)

print("Done.")


🔍 Searching for grants...
🌎 Detected mode: GENERIC
🔍 Total queries generated: 79

🔍 Query: global scholarship for decarbonization research

🔍 Query: nanoporous frameworks research funding worldwide

🔍 Query: ccu research funding worldwide

🔍 Query: global scholarship for ai research

🔍 Query: international nanoporous frameworks travel grant

🔍 Query: international energy travel grant

🔍 Query: international sustainable development goals travel grant

🔍 Query: international mofs travel grant

🔍 Query: grant to present sustainable development goals research

🔍 Query: India researcher international travel funding

🔍 Query: global scholarship for sustainable development goals research

🔍 Query: carbon capture and utilization research funding worldwide

🔍 Query: international ccu travel grant

🔍 Query: travel scholarship for researchers from United Arab Emirates

🔍 Query: grant to present catalysis research

🔍 Query: grant to present carbon capture and utilization research

🔍 Query: global 

In [28]:
WORKDIR = Path.cwd()
with open(WORKDIR / "grant_summary.json", "w") as f:
    json.dump(summaries, f, indent=2)

with open(WORKDIR / "proposal_draft.txt", "w") as f:
    f.write(draft)

print("📁 Files saved:")
print(" - grant_summary.json")
print(" - proposal_draft.txt")


📁 Files saved:
 - grant_summary.json
 - proposal_draft.txt


In [29]:

memory_path = WORKDIR / "memory_bank.json"

record = {
    "timestamp": time.time(),
    "top_url": top.get("url"),
    "score": top.get("score"),
    "org": org_profile["name"]
}

if memory_path.exists():
    data = json.load(open(memory_path))
else:
    data = []

data.append(record)

json.dump(data, open(memory_path, "w"), indent=2)

print("🧠 Memory updated:", memory_path)


🧠 Memory updated: /content/memory_bank.json
